In [1]:
def get_training_chunks(num_services, granularity):
    if granularity is None:
        return [list(range(num_services))]
    # Create chunks of size 'granularity'
    return [list(range(i, min(i + granularity, num_services)))
            for i in range(0, num_services, granularity)]

In [2]:
from typing import List, Optional

import gpytorch
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader

from agent.components.commons import ServiceFeatureMapping, ServiceType
from notebooks.create_deepGP import ServiceGP, prepare_chained_data


# --- 2. MODEL DEFINITION ---


class DynamicServiceChain(torch.nn.Module):
    def __init__(self, service_configs: List[ServiceFeatureMapping], num_inducing: int = 64):
        super().__init__()
        self.configs = service_configs
        self.gp_layers = torch.nn.ModuleList()
        self.likelihoods = torch.nn.ModuleList()

        for i, config in enumerate(service_configs):
            input_dims = len(config.feature_indices)
            if i > 0:
                input_dims += 1  # Previous service output

            gp = ServiceGP(input_dims=input_dims, num_inducing=num_inducing)
            likelihood = gpytorch.likelihoods.GaussianLikelihood()
            self.gp_layers.append(gp)
            self.likelihoods.append(likelihood)

    def forward(self, x, boundary_indices: List[int]):
        dists = []
        last_output = None

        for i, gp in enumerate(self.gp_layers):
            indices = self.configs[i].feature_indices
            current_input = x[:, indices]

            if last_output is not None:
                # DETACH logic: If 'i' is the start of a new chunk,
                # we block the gradient from flowing back to 'i-1'.
                if i in boundary_indices:
                    inp_sample = last_output.detach()
                else:
                    inp_sample = last_output

                current_input = torch.cat([current_input, inp_sample], dim=-1)

            dist = gp(current_input)
            dists.append(dist)
            last_output = dist.rsample().unsqueeze(-1)

        return tuple(dists)


# --- 4. EXECUTION ---

def run_training(granularity: Optional[int] = None):
    # Setup
    torch.set_default_dtype(torch.float64)

    # Dummy data creation for standalone functionality
    # (In real usage, replace with your RASK.preprocess_data(raw_df))
    data = {'max_tp': np.random.rand(90), 'cores': np.random.rand(90),
            'data_quality': np.random.rand(90), 'model_size': np.random.rand(90)}
    df = pd.DataFrame(data)

    QR_MAP = ServiceFeatureMapping(ServiceType.QR, [0, 1])
    CV_MAP = ServiceFeatureMapping(ServiceType.CV, [2, 3, 4])
    PC_MAP = ServiceFeatureMapping(ServiceType.PC, [5, 6])

    chain_definition = ([QR_MAP] * 2) + ([CV_MAP] * 2) + ([PC_MAP] * 2)
    num_services = len(chain_definition)

    train_loader, test_x, test_y, _ = prepare_chained_data(df, chain_definition, 0.2)
    model = DynamicServiceChain(chain_definition)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    mlls = [gpytorch.mlls.VariationalELBO(model.likelihoods[i], model.gp_layers[i],
                                          num_data=len(train_loader.dataset)) for i in range(num_services)]

    # Example: granularity=2, num_services=6
    # chunks = [[0, 1], [2, 3], [4, 5]]
    # boundaries = [2, 4] (Indices where we cut the gradient)

    chunks = get_training_chunks(num_services, granularity)
    boundaries = [chunk[0] for chunk in chunks if chunk[0] != 0]

    print(f"Starting training with granularity={granularity}. Chunks: {chunks}")

    model.train()
    for epoch in range(500):
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()

            # 1. Forward pass tells the model where to break the chain
            distributions = model(x_batch, boundary_indices=boundaries)

            # 2. Backward passes for each chunk
            for chunk_indices in chunks:
                # This loss ONLY sees the graph for its specific chunk
                loss = sum([-mlls[i](distributions[i], y_batch[:, i]) for i in chunk_indices])

                # TODO: try if this makes a difference
                # Since chunks are detached from each other, they don't share paths.
                # You likely don't even need retain_graph=True now!
                loss.backward(retain_graph=True)

            optimizer.step()

        print(f"Final loss: {loss.item():.4f}")

        return model, test_x, test_y


In [4]:

#TODO: That loss is not meaningful, because I must get the RMSE for a prediction across them

model, tx, ty = run_training(granularity=None)

model, tx, ty = run_training(granularity=3)

model, tx, ty = run_training(granularity=2)

model, tx, ty = run_training(granularity=1)

Starting training with granularity=None. Chunks: [[0, 1, 2, 3, 4, 5]]


/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15)

Final loss: 8.0351
Starting training with granularity=3. Chunks: [[0, 1, 2], [3, 4, 5]]


/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15)

Final loss: 3.8152
Starting training with granularity=2. Chunks: [[0, 1], [2, 3], [4, 5]]


/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15)

Final loss: 2.4968
Starting training with granularity=1. Chunks: [[0], [1], [2], [3], [4], [5]]


/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (15)

Final loss: 1.2676
